# Notebook 6 - Portfolio Analysis

this is the main analysis notebook. using all the cleaned data from the earlier notebooks we compute:

- portfolio value over time
- number of positions each quarter
- concentration metrics (what % of the portfolio is in top 5 / top 10 positions)
- HHI (herfindahl hirschman index) - a measure of how concentrated the portfolio is, 1.0 means 100% in one stock
- quarterly turnover - how much of the portfolio changed hands
- largest holdings, biggest buys/sells, new positions, full exits

this is basically what you would see in an institutional research report about a fund manager

In [1]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

working directory: /content/13F-Analytics


In [2]:
from src import analytics, config
from src.utils import load_parquet

holdings = load_parquet(config.HOLDINGS_PARQUET)
tx = load_parquet(config.TRANSACTIONS_PARQUET)

summary = analytics.portfolio_summary(holdings, tx)
summary

,quarter,report_date,portfolio_value_usd,n_positions,top5_weight,top10_weight,hhi,avg_position_usd,median_position_usd,largest_holding,largest_holding_weight,gross_buys_usd,gross_sells_usd,turnover
0,2024Q2,2024-06-30,"279,969,062,343.00",41,0.73,0.90,0.15,"6,828,513,715.68","1,150,430,000.00",APPLE INC,0.30,NaN,NaN,NaN
1,2024Q3,2024-09-30,"266,378,900,503.00",40,0.71,0.90,0.13,"6,659,472,512.57","1,106,001,154.50",APPLE INC,0.26,"3,217,074,394.00","27,562,414,210.00",0.01
2,2024Q4,2024-12-31,"267,175,474,249.00",38,0.72,0.90,0.14,"7,030,933,532.87","1,143,632,802.50",APPLE INC,0.28,"2,248,887,301.00","5,326,180,888.00",0.01
3,2025Q1,2025-03-31,"1,106,550,356.00",4,1.00,1.00,0.46,"276,637,589.00","206,894,955.50",NUCOR CORP,0.63,"1,106,528,324.00","267,175,452,217.00",0.01
4,2025Q2,2025-06-30,"257,521,776,925.00",41,0.70,0.87,0.12,"6,281,018,949.39","1,186,820,921.00",APPLE INC,0.22,"256,416,002,972.00","776,403.00",0.00
5,2025Q3,2025-09-30,"267,334,501,955.00",41,0.70,0.87,0.12,"6,520,353,706.22","1,461,978,000.00",APPLE INC,0.23,"5,609,239,361.00","2,057,449,048.00",0.01
6,2025Q4,2025-12-31,"274,160,086,701.00",42,0.71,0.88,0.13,"6,527,621,111.93","1,292,417,438.50",APPLE INC,0.23,"4,800,187,902.00","5,616,032,717.00",0.02
7,2026Q1,2026-03-31,"263,095,703,570.00",29,0.67,0.91,0.12,"9,072,265,640.34","2,232,726,597.00",APPLE INC,0.22,"14,815,732,156.00","19,793,050,308.00",0.06


In [3]:
# largest holdings this quarter (with CUSIP for verification)
latest = summary["quarter"].iloc[-1]
print(f"largest holdings as of {latest}")
analytics.largest_holdings(holdings, latest, n=10)

largest holdings as of 2026Q1


,issuer,cusip,asset_class,value_usd,portfolio_weight
0,APPLE INC,037833100,Common Stock,"57,843,260,493.00",0.22
1,AMERICAN EXPRESS CO,025816109,Common Stock,"45,859,204,536.00",0.17
2,COCA COLA CO,191216100,Common Stock,"30,420,000,000.00",0.12
3,BANK AMERICA CORP,060505104,Common Stock,"25,039,178,044.00",0.10
4,CHEVRON CORPORATION,166764100,Common Stock,"17,457,364,606.00",0.07
5,OCCIDENTAL PETE CORP,674599105,Common Stock,"17,221,193,015.00",0.07
6,ALPHABET INC,02079K305,Common Stock,"15,600,071,913.00",0.06
7,CHUBB LTD SWITZ,H1467J104,Common Stock,"11,162,836,215.00",0.04
8,MOODYS CORP,615369105,Common Stock,"10,762,190,653.00",0.04
9,KRAFT HEINZ CO,500754106,Common Stock,"7,323,527,057.00",0.03


In [4]:
# biggest buys
print(f"biggest buys in {latest}")
analytics.largest_buys(tx, latest, n=10)

biggest buys in 2026Q1


,issuer,cusip,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,ALPHABET INC,02079K305,BUY,"36,403,656.00","10,014,229,467.00","5,585,842,446.00","15,600,071,913.00"
1,DELTA AIR LINES INC,247361702,NEW POSITION,"39,809,456.00","2,646,532,635.00",NaN,"2,646,532,635.00"
2,ALPHABET INC,02079K107,NEW POSITION,"3,585,215.00","1,028,454,775.00",NaN,"1,028,454,775.00"
3,NEW YORK TIMES CO MTN BE,650111107,BUY,"10,080,791.00","916,555,428.00","351,663,948.00","1,268,219,376.00"
4,LENNAR CORP,526057104,BUY,"3,048,692.00","152,215,251.00","724,837,660.00","877,052,911.00"
5,MACYS INC,55616P104,NEW POSITION,"3,038,355.00","54,963,842.00",NaN,"54,963,842.00"
6,LENNAR CORP,526057302,BUY,"56,723.00","2,780,758.00","17,214,818.00","19,995,576.00"


In [5]:
# biggest sells
print(f"biggest sells in {latest}")
analytics.largest_sells(tx, latest, n=10)

biggest sells in 2026Q1


,issuer,cusip,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,BANK AMERICA CORP,060505104,SELL,"-3,671,769.00","-3,412,098,326.00","28,451,276,370.00","25,039,178,044.00"
1,VISA INC,92826C839,FULL EXIT,"-8,297,460.00","-2,910,002,197.00","2,910,002,197.00",NaN
2,CHEVRON CORPORATION,166764100,SELL,"-45,780,506.00","-2,379,766,525.00","19,837,131,131.00","17,457,364,606.00"
3,MASTERCARD INCORPORATED,57636Q104,FULL EXIT,"-3,986,648.00","-2,275,897,610.00","2,275,897,610.00",NaN
4,CONSTELLATION BRANDS INC,21036P108,SELL,"-12,367,110.00","-1,698,546,500.00","1,793,480,000.00","94,933,500.00"
5,UNITEDHEALTH GROUP INC,91324P102,FULL EXIT,"-5,039,564.00","-1,663,610,472.00","1,663,610,472.00",NaN
6,DOMINOS PIZZA INC,25754A201,FULL EXIT,"-3,350,000.00","-1,396,347,000.00","1,396,347,000.00",NaN
7,AON PLC,G0403H108,FULL EXIT,"-3,602,995.00","-1,271,424,876.00","1,271,424,876.00",NaN
8,POOL CORP,73278L105,FULL EXIT,"-3,068,885.00","-702,007,444.00","702,007,444.00",NaN
9,AMAZON COM INC,023135106,FULL EXIT,"-2,276,000.00","-525,346,320.00","525,346,320.00",NaN


In [6]:
# new positions opened this quarter
print(f"new positions in {latest}")
analytics.new_positions(tx, latest, n=10)

new positions in 2026Q1


,issuer,cusip,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,DELTA AIR LINES INC,247361702,NEW POSITION,"39,809,456.00","2,646,532,635.00",NaN,"2,646,532,635.00"
1,ALPHABET INC,02079K107,NEW POSITION,"3,585,215.00","1,028,454,775.00",NaN,"1,028,454,775.00"
2,MACYS INC,55616P104,NEW POSITION,"3,038,355.00","54,963,842.00",NaN,"54,963,842.00"


In [7]:
# positions fully exited this quarter
print(f"full exits in {latest}")
analytics.full_exits(tx, latest, n=10)

full exits in 2026Q1


,issuer,cusip,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,VISA INC,92826C839,FULL EXIT,"-8,297,460.00","-2,910,002,197.00","2,910,002,197.00",NaN
1,MASTERCARD INCORPORATED,57636Q104,FULL EXIT,"-3,986,648.00","-2,275,897,610.00","2,275,897,610.00",NaN
2,UNITEDHEALTH GROUP INC,91324P102,FULL EXIT,"-5,039,564.00","-1,663,610,472.00","1,663,610,472.00",NaN
3,DOMINOS PIZZA INC,25754A201,FULL EXIT,"-3,350,000.00","-1,396,347,000.00","1,396,347,000.00",NaN
4,AON PLC,G0403H108,FULL EXIT,"-3,602,995.00","-1,271,424,876.00","1,271,424,876.00",NaN
5,POOL CORP,73278L105,FULL EXIT,"-3,068,885.00","-702,007,444.00","702,007,444.00",NaN
6,AMAZON COM INC,023135106,FULL EXIT,"-2,276,000.00","-525,346,320.00","525,346,320.00",NaN
7,HEICO CORP NEW,422806208,FULL EXIT,"-1,294,612.00","-326,798,907.00","326,798,907.00",NaN
8,LIBERTY MEDIA CORP DEL,531229755,FULL EXIT,"-3,018,555.00","-297,357,853.00","297,357,853.00",NaN
9,CHARTER COMMUNICATIONS INC N,16119P108,FULL EXIT,"-1,060,882.00","-221,459,118.00","221,459,118.00",NaN


In [8]:
# asset allocation breakdown
print(f"asset allocation in {latest}")
analytics.asset_allocation(holdings, latest)

asset allocation in 2026Q1


,asset_class,value_usd,weight
0,Common Stock,"263,095,703,570.00",1.00
